# LLM as Judge
This is the code used to test the paragraphs via the LLM grading. The LLM was provided the same rubric as human graders. Each essay was inputted indiviudally to reduce bias, and results were stored in JSON file: ll_judge_paragraph_results.json.


In [50]:
import os
import json
import time
import pandas as pd
from google import genai
from dotenv import load_dotenv

In [ ]:
load_dotenv() 

#Initialize the client (Using gemini api key from .env file))
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


In [53]:
# ── Configure Gemini ──────────────────────────────────────────────────────────
# ── Rubric (full text pasted for the LLM) ─────────────────────────────────────
RUBRIC = """
Argument Quality Rubric
Total: 25 points | Each dimension scored 1–5

Dimension 1: Claim Clarity (max 5)
Does the paragraph state a clear, specific, and defensible position?
Score | Anchor
5     | Position is unambiguous, specific, and immediately identifiable.
4     | Position is clear but could be slightly more precise or specific.
3     | A position is present but vague or buried; requires inference to identify.
2     | Position is implied but never explicitly stated.
1     | No discernible position, or the paragraph contradicts itself.

Dimension 2: Quality of Evidence & Support (max 5)
Are claims backed by concrete reasoning, examples, data, or logical grounds?
Score | Anchor
5     | Multiple specific, relevant pieces of evidence or reasoning. Support is concrete, not generic.
4     | Evidence is present and relevant, but could be more specific or developed.
3     | Some support is offered, but vague, anecdotal, or only partially relevant.
2     | Very limited support; mostly assertions without backing.
1     | No evidence or reasoning provided; purely opinion.

Dimension 3: Warrant & Logical Reasoning (max 5)
Is the logical link between evidence and claim made explicit? Does the argument hold together?
Score | Anchor
5     | Reasoning chain is explicit and sound. Clear why evidence supports the claim. No logical leaps.
4     | Reasoning is mostly clear; minor gaps in the logical chain.
3     | Some reasoning present but the connection between evidence and claim is assumed.
2     | Evidence and claim stated but the logical link is absent or weak.
1     | No logical connection; reasoning is circular, fallacious, or incoherent.

Dimension 4: Counterargument Engagement (max 5)
Does the paragraph acknowledge and address an opposing view or objection?
Score | Anchor
5     | A specific, steelmanned counterargument is acknowledged and meaningfully rebutted or qualified.
4     | A counterargument is acknowledged and addressed, but not fully developed.
3     | An opposing view is briefly mentioned but not engaged with substantively.
2     | A counterargument is vaguely gestured at with no engagement.
1     | No acknowledgment of any opposing view.

Dimension 5: Overall Persuasiveness & Coherence (max 5)
Taken as a whole, how structured, convincing, and rhetorically effective is the paragraph?
Score | Anchor
5     | Argument flows logically, builds to a convincing conclusion, and would persuade a neutral reader.
4     | Mostly persuasive and coherent with minor structural weaknesses.
3     | Argument is followable but disjointed or only partially convincing.
2     | Argument is difficult to follow; structure is unclear or contradictory.
1     | Incoherent or unconvincing as a whole; the paragraph does not function as an argument.

Grader instructions:
- Grade each dimension independently. Do not let overall impression affect individual scores.
- Do NOT consider grammar or writing style unless it directly impedes understanding the argument.
- Use only whole numbers 1–5. No half-scores.
"""

# ── Scoring prompt template ───────────────────────────────────────────────────
SCORE_PROMPT_TEMPLATE = """You are an expert argumentation evaluator using the Toulmin model.

{rubric}

Score the following paragraph on each of the five dimensions using ONLY whole numbers 1–5.
Do not consider grammar or writing style — assess argumentation only.

Output ONLY a valid JSON object (no markdown, no extra text) with these exact keys:
{{
  "claim_clarity": <int 1-5>,
  "claim_clarity_justification": "<one sentence>",
  "evidence_quality": <int 1-5>,
  "evidence_quality_justification": "<one sentence>",
  "warrant_reasoning": <int 1-5>,
  "warrant_reasoning_justification": "<one sentence>",
  "counterargument": <int 1-5>,
  "counterargument_justification": "<one sentence>",
  "overall_persuasiveness": <int 1-5>,
  "overall_persuasiveness_justification": "<one sentence>",
  "total": <int, sum of all five scores>
}}

Paragraph to evaluate:
\"\"\"
{paragraph}
\"\"\"
"""


In [ ]:
df_scores_essay = pd.DataFrame()

def evaluate_and_store(id_number, text):
    global df_scores_essay
    # Generate the prompt
    prompt = SCORE_PROMPT_TEMPLATE.format(rubric=RUBRIC, paragraph=text)
    
    # Call Gemini with JSON mode enabled
    response = client.models.generate_content(
        #Using gemini 3 model, openai key ran out of credits
        model="gemini-3-flash-preview",
        contents=prompt,
        config={'response_mime_type': 'application/json'}
    )
    
    # Parse the output
    scores = json.loads(response.text)
    
    # Calculate the total score in Python for 100% accuracy
    total = sum([
        scores.get("claim_clarity", 0),
        scores.get("evidence_quality", 0),
        scores.get("warrant_reasoning", 0),
        scores.get("counterargument", 0),
        scores.get("overall_persuasiveness", 0)
    ])
    
    # Merge everything into a single dictionary row
    row_data = {
        "id_number": id_number,
        "paragraph": text,
        **scores,  # Expands all the scores and justifications from the JSON
        "total_score": total
    }
    
    # Append to the DataFrame
    df_scores_essay = pd.concat([df_scores_essay, pd.DataFrame([row_data])], ignore_index=True)
    return row_data


In [96]:
#Calling the method for each paragraph pasted
example_id = "9_ai"
example_text = ("""Is adult hippocampal neurogenesis necessary to recover from stressed induced depression?
Chronic stress dysregulates the HPA axis, leading to persstent cortisol release that damages the hippocampus, a region vital for stress regulation via its high density of glucocorticoid receptors. 

The neurogenesis hypothesis explains the delayed effiacy of antidepressants: while neurotransmitter levels change quickly, clinical recovery likely requires downstream BDNF activation to generate new hippocampal neurons.The neurogenesis hypothesis helps explain why antidepressants, often take weeks to improve a patient mood. This delay suggests a downstream mechanism: many antidepressants activate BDNF receptors, which trigger adult hippocampal neurogenesis. However, the necessity of neurogenesis remains a point of debate. Counter-evidence suggests that some patients show recovery through synaptic plasticity rather than the birth of new neurons. Furthermore, animal studies have shown that blocking neurogenesis doesn't always prevent antidepressants from working in every behavioral test. Therefore, while hippocampal neurogenesis is a primary driver of recovery, it may be part of a broader network of structural repairs required to reverse stress-induced depression.
""")
print("Grading in progress...")
evaluate_and_store(example_id, example_text)

#To see the results
print("\n--- Current DataFrame ---")
print(df_scores_essay[['id_number', 'claim_clarity', 'counterargument', 'total_score']])

Grading in progress...

--- Current DataFrame ---
  id_number  claim_clarity  counterargument  total_score
0      9_ai              5                5           25


In [97]:
df_scores_essay[['id_number', 'claim_clarity', 'counterargument', 'total_score']]

,id_number,claim_clarity,counterargument,total_score
0,9_ai,5,5,25


In [ ]:
#Saving the grading as a JSON file:
df_scores_essay.to_json("ll_judge_paragraph_results.json", orient="records", indent=4)

print("File 'll_judge_paragraph_results.json' has been created ")
#Note, due to api key having time-limits some paragraphs were done later (had to wait one minute or so) and then JSON file was appended all together

File 'll_judge_paragraph_results.json' has been created 


In [100]:
df_loaded = pd.read_json("ll_judge_paragraph_results.json", orient="records")

#Show the fullDataFrame
df_loaded

,id_number,paragraph,claim_clarity,claim_clarity_justification,evidence_quality,evidence_quality_justification,warrant_reasoning,warrant_reasoning_justification,counterargument,counterargument_justification,overall_persuasiveness,overall_persuasiveness_justification,total,total_score
0,1_no_ai,Is social media doing more harm than good?\nDe...,4,The paragraph clearly states its position that...,2,"Support mainly consists of broad assertions, l...",3,The reasoning connects the isolation from soci...,4,A specific counterargument about fake informat...,3,The argument is followable but only partially ...,16,16
1,1_ai_gemini_3,Is social media doing more harm than good?\nDe...,4,The paragraph clearly states its position that...,3,Support relies on general assertions about pro...,3,The connection between the general benefits pr...,4,A specific counterargument about fake informat...,3,The argument is followable with a clear stance...,17,17
2,2_no_ai,Is it better to have 1 or 100 friends?\nIt is ...,5,The claim that it is always better to have 1 f...,4,The argument provides specific conceptual reas...,5,The logical connection between the definition ...,5,A specific counterargument regarding the benef...,5,"The argument is very well-structured, flows lo...",24,24
3,2_ai,Is it better to have 1 or 100 friends?\nIt is ...,5,The paragraph clearly and unambiguously states...,4,The support consists of developed reasoning ab...,5,The logical connection between the benefits of...,5,The paragraph effectively acknowledges and ste...,5,"The argument is well-structured, logically coh...",24,24
4,3_no_ai,Should billionaires exist?\nIt is crazy to sta...,4,The paragraph clearly states the position that...,2,The support consists primarily of assertions (...,3,The reasoning attempts to link the vastness of...,2,The paragraph vaguely acknowledges a general o...,3,The argument is followable and states a clear ...,14,14
5,3_ai,It is crazy to state the billionaires should n...,3,The main claim that billionaires are inherentl...,4,The paragraph provides concrete examples like ...,4,The logical chain connecting immense wealth to...,5,The argument effectively acknowledges and rebu...,4,"The argument flows logically, building from th...",20,20
6,4_no_ai,Is it better to have 100 average friends or 1 ...,5,The position favoring a best friend over many ...,3,The support relies on general characteristics ...,4,"The logical link is mostly clear, connecting t...",4,The paragraph acknowledges the opposing view o...,4,The argument is coherent and builds a clear ca...,20,20
7,4_ai,Is it better to have 100 average friends or 1 ...,5,The paragraph states an unambiguous and specif...,3,The support for the claim relies on assertions...,5,"The logical link is explicit and sound, clearl...",5,The argument directly acknowledges and address...,5,"The argument is well-structured, flows logical...",23,23
8,5_no_ai,It is honestly quite embarrassing that in 2026...,5,The position that the British Museum should re...,4,The paragraph provides specific evidence regar...,4,The reasoning mostly connects the evidence to ...,5,A specific counterargument about the safety of...,5,"The argument is structured logically, presents...",23,23
9,5_ai,It is honestly quite embarrassing that in 2026...,5,The paragraph clearly states a specific and un...,5,The argument uses multiple specific pieces of ...,5,The reasoning explicitly connects the evidence...,5,The paragraph acknowledges and directly rebuts...,5,The argument flows logically from a clear clai...,25,25
